<a href="https://colab.research.google.com/github/Karthikreddy1010/Electric-poles-and-wires-detection/blob/main/Yolov8_onlypole_largermodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/pole_data.zip

In [1]:
!pip install ultralytics

from IPython.display import Image
import ultralytics
ultralytics.checks()

Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 41.8/235.7 GB disk)


In [3]:
# ── Install dependencies (run this cell first in Colab) ──────────────
# !pip install ultralytics albumentations opencv-python pyyaml -q

import os
import cv2
import random
import shutil
import yaml
import numpy as np
from pathlib import Path
import albumentations as A
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────

BASE_DIR        = Path("/content/pole_data")

TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"

# Albumentations output (augmented training images)
AUG_TRAIN_IMG   = BASE_DIR / "train_augmented" / "images"
AUG_TRAIN_LBL   = BASE_DIR / "train_augmented" / "labels"

# data.yaml paths
ORIGINAL_YAML   = BASE_DIR / "data.yaml"
AUG_YAML        = BASE_DIR / "data_augmented.yaml"

# ── YOLOv8 settings ──────────────────────────────────────────────────
MODEL_WEIGHTS   = "yolov8x.pt"        # ← UPGRADED to Extra-Large model
PROJECT_NAME    = "/content/gsv_pole_detection"
RUN_NAME        = "exp1_xlarge"
EPOCHS          = 100
IMG_SIZE        = 640
BATCH_SIZE      = 8                    # Reduce if GPU OOM (X model is heavy)
DEVICE          = 0                    # 0 = GPU (T4), "cpu" = CPU
WORKERS         = 2
AUG_COPIES      = 2                    # augmented copies per image → 3x dataset


# ─────────────────────────────────────────────────────────────────────
# STEP 0 — Fix data.yaml: use train as val (no separate val set)
# ─────────────────────────────────────────────────────────────────────

def fix_data_yaml():
    """
    Rewrites data.yaml with absolute Colab paths.
    Since there is no validation split, val is pointed at the
    augmented train set so YOLOv8 can monitor training loss.
    For a proper experiment, consider carving out 10-15% of train
    as a held-out validation split.
    """
    if not ORIGINAL_YAML.exists():
        raise FileNotFoundError(f"data.yaml not found at: {ORIGINAL_YAML}")

    with open(ORIGINAL_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(TRAIN_IMG_DIR)
    data["val"]   = str(TRAIN_IMG_DIR)   # No val split → reuse train for monitoring
    data["test"]  = str(TEST_IMG_DIR)
    data["nc"]    = 1
    data["names"] = ["Pole"]

    with open(ORIGINAL_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"[Step 0] data.yaml fixed with absolute paths:")
    print(f"  train → {TRAIN_IMG_DIR}")
    print(f"  val   → {TRAIN_IMG_DIR}  (no separate val set)")
    print(f"  test  → {TEST_IMG_DIR}")
    print()
    print("  ⚠  NOTE: val is set to train because no validation split exists.")
    print("     For proper evaluation, split 10-15% of train images into a")
    print("     separate val/ folder before training.")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — ALBUMENTATIONS PIPELINE
# ─────────────────────────────────────────────────────────────────────

def get_albumentations_pipeline():
    return A.Compose(
        [
            # Lighting & Contrast
            A.RandomBrightnessContrast(p=0.5),
            A.CLAHE(clip_limit=4.0, p=0.3),

            # Noise
            A.GaussNoise(p=0.3),

            # Blur (vehicle motion + out-of-focus distant poles)
            A.MotionBlur(blur_limit=5, p=0.2),
            A.GaussianBlur(blur_limit=3, p=0.2),

            # Weather effects
            A.RandomFog(
                fog_coef_range=(0.1, 0.3),
                alpha_coef=0.1,
                p=0.15
            ),
            A.RandomRain(
                slant_range=(-10, 10),
                drop_length=10,
                drop_width=1,
                brightness_coefficient=0.9,
                p=0.1
            ),
            A.RandomSunFlare(
                flare_roi=(0, 0, 1, 0.5),
                src_radius=100,
                p=0.1
            ),
            A.RandomShadow(
                shadow_roi=(0, 0.5, 1, 1),
                num_shadows_limit=(1, 2),
                shadow_dimension=5,
                p=0.2
            ),
        ],
        bbox_params=A.BboxParams(
            format="yolo",
            label_fields=["class_labels"],
            min_visibility=0.3,
        ),
    )


# ─────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────

def read_yolo_labels(label_path):
    bboxes, class_labels = [], []
    if not os.path.exists(label_path):
        return bboxes, class_labels
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls = int(parts[0])
                cx, cy, w, h = map(float, parts[1:])
                bboxes.append([cx, cy, w, h])
                class_labels.append(cls)
    return bboxes, class_labels


def write_yolo_labels(label_path, bboxes, class_labels):
    with open(label_path, "w") as f:
        for cls, bbox in zip(class_labels, bboxes):
            f.write(f"{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — Apply Albumentations to training set
# ─────────────────────────────────────────────────────────────────────

def apply_albumentations_to_dataset(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    os.makedirs(AUG_TRAIN_IMG, exist_ok=True)
    os.makedirs(AUG_TRAIN_LBL, exist_ok=True)

    pipeline    = get_albumentations_pipeline()
    image_paths = (
        sorted(Path(TRAIN_IMG_DIR).glob("*.jpg"))
        + sorted(Path(TRAIN_IMG_DIR).glob("*.jpeg"))
        + sorted(Path(TRAIN_IMG_DIR).glob("*.png"))
    )

    print(f"\n[Step 1] Found {len(image_paths)} training images")
    print(f"         Generating {AUG_COPIES} augmented copies each "
          f"→ {len(image_paths) * (AUG_COPIES + 1)} total images\n")

    for idx, img_path in enumerate(image_paths):
        img_path = Path(img_path)
        stem     = img_path.stem
        ext      = img_path.suffix
        lbl_path = TRAIN_LBL_DIR / (stem + ".txt")

        bboxes, class_labels = read_yolo_labels(str(lbl_path))

        image = cv2.imread(str(img_path))
        if image is None:
            print(f"  [WARN] Could not read {img_path.name}, skipping.")
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Save original into augmented folder
        cv2.imwrite(
            str(AUG_TRAIN_IMG / (stem + ext)),
            cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        )
        if lbl_path.exists():
            shutil.copy(str(lbl_path), str(AUG_TRAIN_LBL / (stem + ".txt")))
        else:
            open(str(AUG_TRAIN_LBL / (stem + ".txt")), "w").close()

        # Save augmented copies
        for i in range(AUG_COPIES):
            try:
                result   = pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
                aug_stem = f"{stem}_aug{i + 1}"
                cv2.imwrite(
                    str(AUG_TRAIN_IMG / (aug_stem + ext)),
                    cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR)
                )
                write_yolo_labels(
                    str(AUG_TRAIN_LBL / (aug_stem + ".txt")),
                    result["bboxes"],
                    result["class_labels"]
                )
            except Exception as e:
                pass

        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(image_paths)} images...")

    total = len(list(AUG_TRAIN_IMG.glob("*.*")))
    print(f"\n  Done. Total images in train_augmented/images: {total}")


# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Create data_augmented.yaml
# ─────────────────────────────────────────────────────────────────────

def create_augmented_yaml():
    with open(ORIGINAL_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(AUG_TRAIN_IMG)
    data["val"]   = str(AUG_TRAIN_IMG)   # No separate val — reuse augmented train
    data["test"]  = str(TEST_IMG_DIR)
    data["nc"]    = 1
    data["names"] = ["Pole"]

    with open(AUG_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"\n[Step 2] data_augmented.yaml created at:\n  {AUG_YAML}")
    print(f"  train → {AUG_TRAIN_IMG}")
    print(f"  val   → {AUG_TRAIN_IMG}  (no separate val set)")
    print(f"  test  → {TEST_IMG_DIR}")


# ─────────────────────────────────────────────────────────────────────
# STEP 3 — YOLOv8x Training
# ─────────────────────────────────────────────────────────────────────

def train_yolov8():
    model = YOLO(MODEL_WEIGHTS)

    print(f"\n[Step 3] Starting YOLOv8 training...")
    print(f"  Model    : {MODEL_WEIGHTS}  (Extra-Large — 68.2M params)")
    print(f"  Epochs   : {EPOCHS}")
    print(f"  Img size : {IMG_SIZE}")
    print(f"  Batch    : {BATCH_SIZE}")
    print(f"  ⚠  If GPU OOM, reduce BATCH_SIZE to 4 at the top of the script\n")

    results = model.train(
        data=str(AUG_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        project=PROJECT_NAME,
        name=RUN_NAME,
        exist_ok=True,

        # ── HSV Color Jitter ─────────────────────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.5,

        # ── Geometric Augmentations ──────────────────────────────────
        degrees=4.0,
        translate=0.1,
        scale=0.5,
        shear=1.5,
        perspective=0.0003,

        # ── Flip ─────────────────────────────────────────────────────
        flipud=0.0,         # DISABLED — GSV always upright
        fliplr=0.5,         # poles appear on both sides of road

        # ── Composite Augmentations ──────────────────────────────────
        mosaic=1.0,
        mixup=0.12,
        copy_paste=0.25,

        # ── Occlusion ────────────────────────────────────────────────
        erasing=0.3,

        # ── Mosaic Scheduling ────────────────────────────────────────
        close_mosaic=10,

        # ── Optimizer ────────────────────────────────────────────────
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,

        # ── Output ───────────────────────────────────────────────────
        save=True,
        save_period=10,
        plots=True,
        verbose=True,
        val=True,
        patience=30,        # Early stopping (reduced — no separate val set)
    )
    return results


# ─────────────────────────────────────────────────────────────────────
# STEP 4 — Evaluate on Test Set
# ─────────────────────────────────────────────────────────────────────

def evaluate_model():
    best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
    if not os.path.exists(best_weights):
        print(f"\n[WARN] best.pt not found at {best_weights}. Skipping evaluation.")
        return

    print(f"\n[Step 4] Evaluating on test set...")
    model   = YOLO(best_weights)
    metrics = model.val(
        data=str(AUG_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=PROJECT_NAME,
        name=f"{RUN_NAME}_test_eval",
        verbose=True,
    )

    print("\n  ── Test Set Results ──────────────────────────────")
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    print("  ──────────────────────────────────────────────────")
    return metrics


# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 62)
    print("  YOLOv8x Pole Detection — Colab Training Pipeline")
    print("=" * 62)

    required = {
        "train/images" : TRAIN_IMG_DIR,
        "train/labels" : TRAIN_LBL_DIR,
        "test/images"  : TEST_IMG_DIR,
        "test/labels"  : TEST_LBL_DIR,
    }
    all_ok = True
    print()
    for name, path in required.items():
        exists = path.exists()
        print(f"  [{'OK' if exists else 'MISSING'}] {name}")
        print(f"        → {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise FileNotFoundError(
            "\n[ERROR] One or more folders are missing.\n"
            "Expected structure:\n"
            "  /content/pole_data/\n"
            "  ├── train/images/\n"
            "  ├── train/labels/\n"
            "  ├── test/images/\n"
            "  ├── test/labels/\n"
            "  └── data.yaml\n"
        )

    print()
    fix_data_yaml()                     # Step 0 — Fix paths in data.yaml
    apply_albumentations_to_dataset()   # Step 1 — Albumentations pre-processing
    create_augmented_yaml()             # Step 2 — Create data_augmented.yaml
    train_yolov8()                      # Step 3 — YOLOv8x training
    evaluate_model()                    # Step 4 — Test set evaluation

    print("\n  ═══════════════════════════════════════════════════")
    print("  All done! Your model is ready.")
    print("  ═══════════════════════════════════════════════════")
    print(f"  Best weights : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
    print(f"  Last weights : {PROJECT_NAME}/{RUN_NAME}/weights/last.pt")
    print(f"  Plots/results: {PROJECT_NAME}/{RUN_NAME}/")
    print("\n  To use your model:")
    print("    from ultralytics import YOLO")
    print(f"    model = YOLO('{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')")
    print("    results = model.predict('image.jpg')")

  YOLOv8x Pole Detection — Colab Training Pipeline

  [OK] train/images
        → /content/pole_data/train/images
  [OK] train/labels
        → /content/pole_data/train/labels
  [OK] test/images
        → /content/pole_data/test/images
  [OK] test/labels
        → /content/pole_data/test/labels

[Step 0] data.yaml fixed with absolute paths:
  train → /content/pole_data/train/images
  val   → /content/pole_data/train/images  (no separate val set)
  test  → /content/pole_data/test/images

  ⚠  NOTE: val is set to train because no validation split exists.
     For proper evaluation, split 10-15% of train images into a
     separate val/ folder before training.

[Step 1] Found 5522 training images
         Generating 2 augmented copies each → 16566 total images



/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


  Processed 50/5522 images...
  Processed 100/5522 images...
  Processed 150/5522 images...
  Processed 200/5522 images...
  Processed 250/5522 images...
  Processed 300/5522 images...
  Processed 350/5522 images...
  Processed 400/5522 images...
  Processed 450/5522 images...
  Processed 500/5522 images...
  Processed 550/5522 images...
  Processed 600/5522 images...
  Processed 650/5522 images...
  Processed 700/5522 images...
  Processed 750/5522 images...
  Processed 800/5522 images...
  Processed 850/5522 images...
  Processed 900/5522 images...
  Processed 950/5522 images...
  Processed 1000/5522 images...
  Processed 1050/5522 images...
  Processed 1100/5522 images...
  Processed 1150/5522 images...
  Processed 1200/5522 images...
  Processed 1250/5522 images...
  Processed 1300/5522 images...
  Processed 1350/5522 images...
  Processed 1400/5522 images...
  Processed 1450/5522 images...
  Processed 1500/5522 images...
  Processed 1550/5522 images...
  Processed 1600/5522 images

KeyboardInterrupt: 